# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a walkthrough for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset is FAIR² certified and follows the Croissant schema, available at the given URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
md = dataset.metadata
print(f"Dataset Name: {md.name}\n\nDescription: {md.description}\n")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

We list the record sets and, for each, their fields and columns, referencing all entities by their `@id`.

In [ ]:
# List available record sets and their fields by @id
recordsets = list(dataset.record_sets)
print(f"Found {len(recordsets)} record set(s) in the dataset.\n")
record_set_ids = []
for rs in recordsets:
    print(f"RecordSet Name: {rs.name}")
    print(f"  @id: {rs.id}")
    record_set_ids.append(rs.id)
    if hasattr(rs, 'fields') and rs.fields:
        print("  Fields:")
        for f in rs.fields:
            print(f"    - {getattr(f, 'name', '')} (@id: {getattr(f, 'id', '<none>')})")
    if hasattr(rs, 'columns') and rs.columns:
        print("  Columns:")
        for c in rs.columns:
            print(f"    - {getattr(c, 'name', '')} (@id: {getattr(c, 'id', '<none>')})")
    print()

## 3. Data Extraction
Load data from each record set into a pandas DataFrame using its `@id`.

> **Note**: The code below dynamically loads all available record sets (using their `@id`), collects the data into a dictionary of DataFrames, and prints available columns for the first record set.

In [ ]:
# Extract data for all record sets by @id
dataframes = dict()

# We'll load at most 5 rows for demonstration/preview for each record set
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} records for RecordSet '@id': {rs_id}")
        print(f"Available columns: {list(df.columns)}\n")
    else:
        print(f"No records found for RecordSet '@id': {rs_id}")
        dataframes[rs_id] = pd.DataFrame([])

# Preview for the first non-empty record set
main_record_set_id = None
for k, v in dataframes.items():
    if not v.empty:
        main_record_set_id = k
        break
if main_record_set_id is not None:
    print(f"First five records of RecordSet '@id': {main_record_set_id}:")
    display(dataframes[main_record_set_id].head())
else:
    print("No data found in any record set.")

## 4. Exploratory Data Analysis (EDA)
Apply basic EDA steps such as filtering, normalization, and grouping. All field and column accesses must reference the `@id` of the entities as per schema documentation.

In [ ]:
# Identify a numeric field (by @id) in main record set for demonstration
df = dataframes.get(main_record_set_id, pd.DataFrame([])).copy()
if df.empty:
    print(f"No data available for EDA in record set '@id': {main_record_set_id}")
else:
    # Try to find a numeric column (float/int) for demo. If known, set @id here, otherwise grab first float/integer.
    import numpy as np
    numeric_field_id = None
    for c in df.columns:
        if pd.api.types.is_numeric_dtype(df[c]):
            numeric_field_id = c
            break
    if numeric_field_id is None:
        print("No numeric field detected in the main record set.\n")
    else:
        print(f"Using numeric field '@id': {numeric_field_id}\n")
        # Example: filter for values above the 75th percentile
        threshold = df[numeric_field_id].quantile(0.75)
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (75th percentile): {len(filtered_df)} rows\n")
        # Normalize this numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        print("Preview of normalized values:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        # If another non-numeric field exists, group by that
        group_field_id = None
        for c in df.columns:
            if c != numeric_field_id and not pd.api.types.is_numeric_dtype(df[c]):
                group_field_id = c
                break
        if group_field_id:
            print(f"\nGrouping filtered data by '@id': {group_field_id}")
            grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(grouped.head())
        else:
            print("No suitable categorical/grouping field detected.")

## 5. Visualization
Visualize the distribution of a selected numeric field and, if available, its values grouped by a categorical variable. All axis labels or titles should identify fields by `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df.empty or numeric_field_id is None:
    print("No data available for visualization.")
else:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30, color='dodgerblue')
    plt.title(f"Distribution of Field: '@id' {numeric_field_id}")
    plt.xlabel(f"@id: {numeric_field_id}")
    plt.ylabel("Count")
    plt.show()

    if group_field_id:
        plt.figure(figsize=(8, 5))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id} (@id)")
        plt.xlabel(f"@id: {group_field_id}")
        plt.ylabel(f"@id: {numeric_field_id}")
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to:
- Load and explore Croissant-compliant datasets using `mlcroissant`, referencing all entities by their `@id` field for reproducibility.
- Identify record sets and fields for deeper data analysis.
- Filter and normalize numeric fields, and group by categorical fields.
- Visualize distributions and relationships between variables.

For further analysis, consult the dataset documentation and explore additional record sets or fields by their `@id`.